In [ ]:
import os
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)

In [ ]:
pip install open_clip_torch

In [ ]:
pip install git+https://github.com/modestyachts/ImageNetV2_pytorch

In [ ]:
import torch
import open_clip
from imagenetv2_pytorch import ImageNetV2Dataset
from torch.utils.data import DataLoader

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Load the model, preprocessing function and tokenizer

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

### Load the ImageNet-V2 dataset

In [ ]:
dataset = ImageNetV2Dataset("matched-frequency", location= "../data", transform=preprocess) # supports matched-frequency, threshold-0.7, top-images variants
dataloader = DataLoader(dataset, batch_size=32, num_workers = 2) # use whatever batch size you wish

In [ ]:
for batch_idx, (image, label) in enumerate(dataloader):
    print(f"Batch ID: {batch_idx} Image Shape: {image.shape}, Label Shape: {label.shape}")
    break

In [ ]:
from src.clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

### Build the text features

In [ ]:
from src.imagenet_classes import IMAGENET_CLASS_NAMES, IMAGENET_TEMPLATES

In [ ]:
print(f"{len(IMAGENET_CLASS_NAMES)}, {len(IMAGENET_TEMPLATES)}")

In [ ]:
text_features = build_and_cache_text_features(model, tokenizer, IMAGENET_CLASS_NAMES, IMAGENET_TEMPLATES, device, "../features", "imagenet1k_text_features")

### Build the Image Features

In [ ]:
image_features_and_labels = build_and_cache_image_features(model, device, dataloader, "../features", "imagenetv2")

### Zero Shot Evaluation

Top-1 accuracy: 57.83, 
Top-5 accuracy: 85.03

In [ ]:
image_features = image_features_and_labels['image_features'].to(device)
labels = image_features_and_labels['labels'].to(device)

similarity = image_features @ text_features
logits = 100 * similarity
acc1, acc5 = top_k_accuracy(logits, labels, topk=(1, 5))

In [ ]:
n = len(labels)

top1 = (acc1 / n) * 100
top5 = (acc5 / n) * 100

print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")

### Test the cached features

In [ ]:
features = load_cached_features("../features/imagenetv2.pt", "../features/imagenet1k_text_features.pt")

In [ ]:
image_features = features["image_features"]
labels = features["labels"]
text_features = features["text_features"]

similarity = image_features @ text_features
logits = 100 * similarity
acc1, acc5 = top_k_accuracy(logits, labels, topk=(1, 5))

In [ ]:
n = len(labels)

top1 = (acc1 / n) * 100
top5 = (acc5 / n) * 100

print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")